<a href="https://colab.research.google.com/github/donggyu3/bigdata/blob/main/week05_5756237_%EA%B9%80%EB%8F%99%EA%B7%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5주차 과제
빅데이터프로그래밍 · 통계학과

**이름: 김동규**
**학번: 5756237**

## 제출 방법
모든 셀을 실행해 출력이 보이는 상태로 저장한 뒤 `week05_학번_이름.ipynb` 로 제출합니다.


In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(0)


## 문제 1. 사용자 정의 Dataset 작성 (40점)
아래 조건으로 Dataset 클래스를 작성하세요.

- 데이터 개수는 30개, 입력 특성은 4개
- `__init__` `__len__` `__getitem__` 세 메서드를 모두 구현
- 각 메서드가 하는 일을 주석으로 한 줄씩 설명


In [2]:
# 답안
import torch
from torch.utils.data import Dataset

class CustomDataset(Dataset):
    def __init__(self):
        # 1. 데이터를 초기화하고 30개의 데이터, 4개의 입력 특성을 가지는 random Tensor 및 정답 Tensor 생성
        self.x_data = torch.randn(30, 4)
        self.y_data = torch.randn(30, 1)

    def __len__(self):
        # 2. 전체 데이터셋의 개수(행의 수, 30개)를 반환
        return len(self.x_data)

    def __getitem__(self, idx):
        # 3. 인덱스(idx)에 해당하는 입력 특성 단일 샘플과 정답 단일 샘플을 튜플 형태로 반환
        return self.x_data[idx], self.y_data[idx]


## 문제 2. DataLoader로 batch 출력 (30점)
문제 1의 Dataset을 DataLoader에 넣고 아래를 출력하세요.

- batch 개수
- 각 batch의 입력 shape과 정답 shape
- 마지막 batch의 크기가 다른 이유 (batch size를 7로 두고 확인)


In [3]:
# 답안
from torch.utils.data import DataLoader

# Dataset 및 DataLoader 생성 (batch_size=7)
dataset = CustomDataset()
dataloader = DataLoader(dataset, batch_size=7, shuffle=False)

# 1. 전체 batch 개수 출력
print(f"전체 batch 개수: {len(dataloader)}")
print("-" * 40)

# 2. 각 batch의 입력 shape과 정답 shape 출력
for batch_idx, (batch_x, batch_y) in enumerate(dataloader, start=1):
    print(f"Batch {batch_idx} - 입력 shape: {batch_x.shape}, 정답 shape: {batch_y.shape}")


전체 batch 개수: 5
----------------------------------------
Batch 1 - 입력 shape: torch.Size([7, 4]), 정답 shape: torch.Size([7, 1])
Batch 2 - 입력 shape: torch.Size([7, 4]), 정답 shape: torch.Size([7, 1])
Batch 3 - 입력 shape: torch.Size([7, 4]), 정답 shape: torch.Size([7, 1])
Batch 4 - 입력 shape: torch.Size([7, 4]), 정답 shape: torch.Size([7, 1])
Batch 5 - 입력 shape: torch.Size([2, 4]), 정답 shape: torch.Size([2, 1])


## 문제 3. batch shape 설명 (10점)
문제 2의 출력에서 batch shape이 왜 그렇게 나오는지 설명하세요.
`(batch_size, 특성 수)` 의 각 숫자가 무엇을 뜻하는지 적으면 됩니다.


**답:** batch_size는 한 번의 가중치 갱신 연산 시 동시에 묶어서 입력되는 데이터 샘플의 개수를 뜻하고 특성 수는 데이터 샘플 하나가 가지는 입력 변수의 개수를 뜻하므로,(batch_size, 특성 수) 구조는 각 데이터의 특성들이 배치 크기만큼 행으로 쌓여 행렬 형태를 이루기 때문에 그렇게 나온다.


## 문제 4. 학습 루프 작성 (20점)
문제 1의 Dataset으로 아래를 수행하세요.

1. `nn.Linear` 모델 하나를 만든다
2. 다섯 단계(예측 → 손실 → 초기화 → 역전파 → 수정)로 학습 루프를 작성한다
3. epoch마다 평균 손실을 출력해 손실이 내려가는지 확인한다


In [4]:
# 답안
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

# 0. 데이터셋 및 데이터로더 준비
dataset = CustomDataset()
dataloader = DataLoader(dataset, batch_size=7, shuffle=True)

# 1. nn.Linear 모델 생성 (입력 특성 4 -> 출력 1)
model = nn.Linear(in_features=4, out_features=1)

# 손실 함수 및 최적화 함수(Optimizer) 정의
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

# 2. 학습 루프 수행 (예: 10 Epochs)
epochs = 10

for epoch in range(1, epochs + 1):
    total_loss = 0.0

    for batch_x, batch_y in dataloader:
        # Step 1: 예측 (Forward pass)
        prediction = model(batch_x)

        # Step 2: 손실 계산 (Loss computation)
        loss = criterion(prediction, batch_y)

        # Step 3: 기울기 초기화 (Zero gradients)
        optimizer.zero_grad()

        # Step 4: 역전파 (Backward pass)
        loss.backward()

        # Step 5: 가중치 수정/갱신 (Optimizer step)
        optimizer.step()

        total_loss += loss.item() * len(batch_x)

    # 3. epoch마다 평균 손실(Average Loss) 출력
    avg_loss = total_loss / len(dataset)
    print(f"Epoch {epoch:2d}/{epochs} - Average Loss: {avg_loss:.4f}")

Epoch  1/10 - Average Loss: 0.8994
Epoch  2/10 - Average Loss: 0.8605
Epoch  3/10 - Average Loss: 0.8376
Epoch  4/10 - Average Loss: 0.8203
Epoch  5/10 - Average Loss: 0.8083
Epoch  6/10 - Average Loss: 0.8045
Epoch  7/10 - Average Loss: 0.7989
Epoch  8/10 - Average Loss: 0.7969
Epoch  9/10 - Average Loss: 0.7960
Epoch 10/10 - Average Loss: 0.7963
